# Proyecto Aprendizaje de Máquina - Clasificación de Textos por Década

## Integrantes

| Nombre             | Código    | Correo electrónico           |
|--------------------|-----------|-----------|
| Adrian Velasquez   | 202222737 | a.velasquezs@uniandes.edu.co |
| Andres Botero Ruiz | 202223503 | a.boteror@uniandes.edu.co    |
| Daniel Vargas López| 202123892 | d.vargasl2@uniandes.edu.co|
| Juan David Torres Albarracín          | 202317608    | jd.torresa1@uniandes.edu.co                   |

# Preparación del entorno de trabajo

En esta celda se importan todas las librerias necesarias para el proyecto, incluyendo manejo de datos, visualizacion, procesamiento de texto, modelos de clasificacion y utilidades de evaluacion. Tambien se configura un estilo visual base para las graficas.

In [6]:
import os
import re
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\a.boteror\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Aqui se cargan los archivos de entrenamiento y evaluacion desde CSV. Al final se imprime el tamano de ambos conjuntos para verificar que los datos se leyeron correctamente.

In [7]:
train_data = pd.read_csv('train.csv')
eval_data = pd.read_csv('eval.csv')
print(f"✓ Train: {train_data.shape} | Eval: {eval_data.shape}")

✓ Train: (31403, 2) | Eval: (3490, 2)


Esta celda realiza una inspeccion rapida del dataset de entrenamiento: numero total de ejemplos, cantidad de decadas distintas y numero de textos duplicados. Es un chequeo inicial de calidad de datos.

In [8]:
print(f"Total: {len(train_data):,} | Décadas: {len(train_data['decade'].unique())} | Duplicados: {train_data['text'].duplicated().sum()}")

Total: 31,403 | Décadas: 39 | Duplicados: 51


# Limpieza y Preparación

En esta seccion se define el preprocesamiento de texto: correccion de artefactos OCR, normalizacion de URLs, numeros, puntuacion y espacios. Luego se limpia el dataset, se elimina texto vacio y se hace la particion train/test estratificada por decada.

In [9]:
# ─── Colapsar artefactos OCR: letras sueltas separadas por espacios ───────────
# Ej: "C  a  s  a" → "Casa"  |  "L  u  z" → "Luz"
def collapse_spaced_chars(text):
    pattern = r'(?<!\w)((?:[A-Za-záéíóúüñÁÉÍÓÚÜÑ]\s{1,2}){2,}[A-Za-záéíóúüñÁÉÍÓÚÜÑ])(?!\w)'
    def join_match(m):
        return re.sub(r'\s+', '', m.group(0))
    return re.sub(pattern, join_match, text)

# ─── Normalización mínima (usada por el modelo) ───────────────────────────────
# Se hace ANTES del colapso OCR para no destruir la señal de caracteres
def normalize_for_model(text):
    text = str(text)
    # OCR fix
    text = collapse_spaced_chars(text)
    # URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)
    # Normalizar números
    text = re.sub(r'\d{4}', ' YEAR ', text)
    text = re.sub(r'\d+', ' NUM ', text)
    # Normalizar puntuación repetida
    text = re.sub(r'([!?.,])\1+', r'\1', text)
    # Espacios
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ─── Preparar dataset unificado ───────────────────────────────────────────────
df_model = train_data.drop_duplicates()
df_model = df_model[df_model['text'].notna()].copy()
df_model['text_model'] = df_model['text'].apply(normalize_for_model)
df_model = df_model[df_model['text_model'].str.len() > 0].copy()

print(f"Datos limpios: {len(df_model):,}")

X_train_char, X_test_char, y_train_char, y_test_char = train_test_split(
    df_model['text_model'], df_model['decade'],
    test_size=0.2, random_state=42, stratify=df_model['decade']
)
print(f"Train: {len(X_train_char):,} | Test: {len(X_test_char):,}")

Datos limpios: 31,369
Train: 25,095 | Test: 6,274


## MODELO: Linear SVM con n-gramas de caracteres

Aqui se construye y entrena el modelo principal (Linear SVM) usando n-gramas de caracteres con TF-IDF. Tambien se configura GridSearchCV para ajustar hiperparametros y evaluar el rendimiento en validacion cruzada y en el conjunto de prueba.

In [10]:
print("\n" + "="*80)
print("MODELO: Linear SVM — char n-grams + word n-grams (FeatureUnion)")
print("="*80)

from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV

feature_union = FeatureUnion([
    ('char', TfidfVectorizer(
        analyzer='char',
        ngram_range=(2, 6),
        min_df=2,
        max_features=1_600_000,
        sublinear_tf=True,
        lowercase=False
    )),
])

pipeline = Pipeline([
    ('features', feature_union),
    ('classifier', LinearSVC(
        max_iter=2000,
        dual=True,
        class_weight='balanced',
        random_state=42
    ))
])

param_grid = {
    'classifier__C': [0.588],
    # 'features__char__min_df': [2],
    # 'features__char__ngram_range': [(2,4), (2,5), (2,6)],
    # 'classifier__loss': ['hinge', 'squared_hinge'],
    # 'features__char__sublinear_tf': [True]
    # 'features__char__norm': ['l2', None],
    # 'features__char__use_idf': [True, False],
}

grid_char_svm = GridSearchCV(pipeline, param_grid, cv=3,
                    scoring='accuracy', n_jobs=-1, verbose=1)

grid_char_svm.fit(X_train_char, y_train_char)

# Guardar modelo joblib
# joblib.dump(grid_char_svm.best_estimator_, 'best_model.joblib')
# print("✓ Modelo guardado: best_model.joblib")

y_pred_char_svm = grid_char_svm.predict(X_test_char)

print(f"✓ Mejores params: {grid_char_svm.best_params_}")
print(f"✓ CV: {grid_char_svm.best_score_:.4f} | Test: {accuracy_score(y_test_char, y_pred_char_svm):.4f}")


MODELO: Linear SVM — char n-grams + word n-grams (FeatureUnion)
Fitting 3 folds for each of 1 candidates, totalling 3 fits
✓ Mejores params: {'classifier__C': 0.588}
✓ CV: 0.2756 | Test: 0.2957


En esta celda se guarda en disco el mejor estimador encontrado durante la busqueda, utilizando formato joblib. Esto permite reutilizar el modelo sin tener que reentrenarlo desde cero.

In [11]:
# Guardar modelo joblib
joblib.dump(grid_char_svm.best_estimator_, 'best_model.joblib')
print("✓ Modelo guardado: best_model.joblib")

✓ Modelo guardado: best_model.joblib


# Exportación del Modelo y Predicciones

Esta celda reentrena el mejor pipeline con todo el conjunto de entrenamiento limpio y luego genera predicciones sobre eval.csv. Finalmente exporta las respuestas al archivo predictions.csv y muestra una vista previa.

In [12]:
# ─── Reentrenar en TODO el dataset y generar predicciones ─────────────────────
eval_model = eval_data.copy()
eval_model['text_eval'] = eval_model['text'].apply(normalize_for_model)
X_eval = eval_model['text_eval']
X_full_train = df_model['text_model']

from sklearn.base import clone
best_model_retrained = clone(grid_char_svm.best_estimator_)
best_model_retrained.fit(X_full_train, df_model['decade'])

y_pred_eval = best_model_retrained.predict(X_eval)

predictions_df = pd.DataFrame({'id': eval_data['id'], 'answer': y_pred_eval})
predictions_df.to_csv('predictions.csv', index=False)
print(f"✓ Predicciones generadas: {len(predictions_df)} samples")
print(f"✓ Archivo 'predictions.csv' creado")
print("\nPrimeras 5 predicciones:")
print(predictions_df.head())

✓ Predicciones generadas: 3490 samples
✓ Archivo 'predictions.csv' creado

Primeras 5 predicciones:
   id  answer
0   0     173
1   1     185
2   2     150
3   3     171
4   4     153
